# Assignment 3: Machine Learning (CSE343/CSE543)
## Instructions
## ✅ Rename the filename with your roll number. E.g. if your roll number is `MT22018` then rename the file `MT22018_a3.ipynb` before submitting.
## ✅ Write code only in the sections marked with `### YOUR CODE HERE`. No, you can NOT write code anywhere else.
## ✅ Do NOT install or import any other libraries beyond what is imported below.
## ❌ Do not modify any other function or class definitions; doing so may lead to the autograder failing to judge your submission, resulting in a zero.
## ❌ Deleting or adding new cells may lead to the autograder failing to judge your submission, resulting in a zero. Even if a cell is empty, do NOT delete it.
## ❌ Do NOT use sklearn for implementing the core algorithms in Q1 (cross-validation loop), Q3 (neural network), or Q4 (SVM). sklearn is allowed only for data generation and metrics.

**Total Marks: 14**
- Q1: K-Fold Cross-Validation (3 marks)
- Q2: Bias-Variance Tradeoff (2 marks)
- Q3: Neural Network from Scratch (5 marks)
- Q4: Support Vector Machine (4 marks)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles, make_blobs, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

---

# Q1: K-Fold Cross-Validation from Scratch (3 marks)

Implement k-fold cross-validation **without using sklearn's KFold or cross_val_score**. You may use sklearn models as classifiers but the splitting and CV loop logic must be yours.

### Part A (1 mark): `create_folds`
Implement a function that splits data indices into k folds. Each sample must appear in exactly one validation fold.

### Part B (1 mark): `run_cv`
Implement a function that performs k-fold CV given a model, data, and k. Returns a dictionary with per-fold scores and summary statistics.

### Part C (1 mark): `create_stratified_folds`
Same as Part A but preserves the class distribution in each fold. For an imbalanced dataset, each fold should have approximately the same class ratio as the full dataset (within ±5%).

# Q1: K-Fold Cross-Validation from Scratch (3 marks)

Implement k-fold cross-validation **without using sklearn's KFold or cross_val_score**. You may use sklearn models as classifiers but the splitting and CV loop logic must be yours.

### Part A (1 mark): `create_folds`
Implement a function that splits data indices into k folds. Each sample must appear in exactly one validation fold.

### Part B (1 mark): `run_cv`
Implement a function that performs k-fold CV given a model, data, and k. Returns a dictionary with per-fold scores and summary statistics.

### Part C (1 mark): `create_stratified_folds`
Same as Part A but preserves the class distribution in each fold. For an imbalanced dataset, each fold should have approximately the same class ratio as the full dataset (within ±5%).

In [ ]:
def create_folds(X, y, k):
    """
    Split dataset indices into k folds.

    Args:
        X: numpy array of shape (n_samples, n_features)
        y: numpy array of shape (n_samples,)
        k: int, number of folds

    Returns:
        folds: list of k tuples (train_indices, val_indices)
               where each is a numpy array of indices
    """
    ### YOUR CODE HERE
    n_samples, n_features = X.shape
    
    result = []
    foldSize = n_samples // k
    remainder = n_samples % k

    valStartIndex = 0
    for i in range(k):
        valEndIndex = valStartIndex + foldSize
        if (remainder > 0):
            valEndIndex += 1
            remainder -= 1

        valIndices = np.asarray(range(valStartIndex, valEndIndex), dtype=int)
        trainIndices = np.concat(  [np.asarray(range(0, valStartIndex), dtype=int), np.asarray(range(valEndIndex, n_samples), dtype=int)] )

        fold = (trainIndices, valIndices)
        result.append(fold)

        valStartIndex = valEndIndex

    return result
    
    raise NotImplementedError()


def run_cv(model_class, X, y, k=5):
    """
    Perform k-fold cross-validation.

    Args:
        model_class: a sklearn-compatible model instance (has fit/predict)
        X: numpy array (n_samples, n_features)
        y: numpy array (n_samples,)
        k: int, number of folds

    Returns:
        dict with keys:
            'fold_scores': list of k float accuracy scores
            'mean_score': float, mean of fold scores
            'std_score': float, std of fold scores
    """
    ### YOUR CODE HERE

    fold_scores = []

    splits = create_folds(X, y, k)

    for i in range(k):
        trainIndices, valIndices = splits[i][0], splits[i][1]
        trainInput = X[trainIndices]
        trainLabel = y[trainIndices]

        valInput = X[valIndices]
        valTrue = y[valIndices].flatten()

        model_class.fit(trainInput, trainLabel)
        valPred = model_class.predict(valInput)

        accuracy = accuracy_score(valTrue, valPred)

        fold_scores.append(accuracy)
    
    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)

    result = {'fold_scores': fold_scores, 'mean_score':mean_score, 'std_score':std_score}
    return result

    raise NotImplementedError()


def create_stratified_folds(X, y, k):
    """
    Split dataset into k stratified folds (class proportions preserved).

    Args:
        X: numpy array (n_samples, n_features)
        y: numpy array (n_samples,)
        k: int, number of folds

    Returns:
        folds: list of k tuples (train_indices, val_indices)
    """
    ### YOUR CODE HERE
    # Dict containing (class : list of indices of samples)
    n_samples, n_features = X.shape

    indicesPerClass = {}
    for i in range(len(y)):
        if y[i] in indicesPerClass:
            indicesPerClass[y[i]].append(i)
        else:
            indicesPerClass[y[i]] = [i]
    
    perClassIndicesInChunks = {}
    for i in indicesPerClass:
        indices = indicesPerClass[i]
        size = len(indices)

        chunks = []
        chunkSize = size // k
        remainder = size % k
        startIndex = 0
        for j in range(k):
            endIndex = startIndex + chunkSize
            if (endIndex > size):
                endIndex = size
            if (remainder > 0):
                endIndex += 1
                remainder -= 1

            chunkIndices = np.asarray(indices[startIndex: endIndex])
            chunks.append(chunkIndices)

            startIndex = endIndex

        perClassIndicesInChunks[i] = chunks
    
    result = []
    for i in range(k):
        valIndices = []
        trainIndices = []
        for j in perClassIndicesInChunks:
            chunks = perClassIndicesInChunks[j]
            valIndices.extend(chunks[i])
            trainIndices.extend(np.concat(chunks[0:i] + chunks[i+1:]))

        valIndices = np.asarray(valIndices, dtype=int)
        trainIndices = np.asarray(trainIndices, dtype=int)

        result.append((trainIndices, valIndices))
    
    return result

In [ ]:
# DO NOT MODIFY — Autograder test for Q1 Part A

In [ ]:
# DO NOT MODIFY — Autograder test for Q1 Part B

In [ ]:
# DO NOT MODIFY — Autograder test for Q1 Part C (Stratified)


# Q2: Bias-Variance Tradeoff (2 marks)

Implement the bootstrap-based bias-variance decomposition for regression.

**Background:** For a regression model, the expected prediction error can be decomposed as:
$\text{Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$

Where:
- $\text{Bias}^2 = \frac{1}{n}\sum_i (\bar{f}(x_i) - y_i)^2$ where $\bar{f}(x_i)$ is the mean prediction across bootstraps
- $\text{Variance} = \frac{1}{n}\sum_i \text{Var}[\hat{f}(x_i)]$ across bootstrap predictions
- $\text{Total Error} = \frac{1}{n}\sum_i \mathbb{E}[(\hat{f}(x_i) - y_i)^2]$ averaged across bootstraps

### Part A (1 mark): `bootstrap_train_predict`
Train a model on multiple bootstrap samples and collect predictions on a fixed test set.

### Part B (1 mark): `compute_bias_variance`
Compute the bias², variance, and total error from the predictions matrix.

After implementing both, plot bias² and variance vs polynomial degree (1 to 15) — this plot is for your understanding and is NOT graded.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline


def bootstrap_train_predict(model_class, X_train, y_train, X_test, n_bootstraps=50):
    """
    Train model on bootstrap samples, predict on fixed test set.

    Args:
        model_class: sklearn model instance (will be cloned for each bootstrap)
        X_train: (n_train, n_features)
        y_train: (n_train,)
        X_test: (n_test, n_features)
        n_bootstraps: int

    Returns:
        predictions: numpy array of shape (n_bootstraps, n_test_samples)
    """
    result = []

    rng = np.random.default_rng()
    n_train, n_features = X_train.shape
    indices = tuple(range(0, n_train))
    for i in range(n_bootstraps):
        sampleIndices = rng.choice(indices, size = n_train)
        trainSample = X_train[sampleIndices]
        trainLabel = y_train[sampleIndices]

        clonedModel = clone(model_class)
        clonedModel.fit(trainSample, trainLabel)
        pred = clonedModel.predict(X_test)
        result.append(pred)

    result = np.asarray(result)
    return result
    raise NotImplementedError()


def compute_bias_variance(predictions, y_test):
    """
    Compute bias-variance decomposition from predictions matrix.

    Args:
        predictions: numpy array of shape (n_bootstraps, n_test_samples)
        y_test: numpy array of shape (n_test_samples,)

    Returns:
        dict with keys:
            'bias_squared': float
            'variance': float
            'total_error': float
    """
    ### YOUR CODE HERE
    mean = np.mean(predictions, axis=0)
    biasSq = np.mean( (mean - y_test) ** 2)
    variance = np.mean(np.var(predictions, axis=0))
    totalError = np.mean( (predictions - y_test)**2)

    result = {'bias_squared':biasSq, 'variance':variance, 'total_error':totalError}
    return result
    raise NotImplementedError()

In [ ]:
# DO NOT MODIFY — Autograder test for Q2 Part A

In [ ]:
# DO NOT MODIFY — Autograder test for Q2 Part B

In [ ]:
# Plot bias-variance tradeoff for polynomial degrees 1-15
# This cell is for your understanding and is NOT graded

### YOUR CODE HERE

# Q3: Neural Network from Scratch (5 marks)

Implement a feedforward neural network using **only NumPy**. No PyTorch, TensorFlow, or sklearn neural network classes allowed.

**Architecture Requirements:**
- Configurable hidden layers via `layer_dims` parameter (e.g., `[2, 64, 32, 1]`)
- ReLU activation for hidden layers
- Sigmoid activation for output layer (binary classification)
- Binary Cross-Entropy loss
- Full-batch gradient descent

**Grading Breakdown:**
1. Weight initialization & shapes (0.5 marks)
2. Forward pass (1 mark)
3. Backpropagation — gradient computation (1.5 marks)
4. Training loop (0.5 marks)
5. Accuracy on `make_moons` dataset (1.5 marks) — bracket grading:

| Accuracy | Score |
|----------|-------|
| > 93%    | 1.5/1.5 (0.5 + 0.5 + 0.5) |
| 88–93%   | 1.0/1.5 (0.5 + 0.5)       |
| 82–88%   | 0.5/1.5 (0.5)             |
| < 82%    | 0/1.5                     |

**Mathematical Reference:**

Forward pass:
$Z^{[l]} = W^{[l]T} A^{[l-1]} + b^{[l]}$
$A^{[l]} = g^{[l]}(Z^{[l]})$

where $g$ is ReLU for hidden layers and Sigmoid for the output layer.

Binary Cross-Entropy Loss:
$\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m}[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$

Backpropagation (output layer):
$dZ^{[L]} = A^{[L]} - Y$
$dW^{[L]} = \frac{1}{m} dZ^{[L]} A^{[L-1]T}$
$db^{[L]} = \frac{1}{m} \sum dZ^{[L]}$

Backpropagation (hidden layers):
$dA^{[l]} = W^{[l+1]} dZ^{[l+1]}$
$dZ^{[l]} = dA^{[l]} * g'^{[l]}(Z^{[l]})$

In [ ]:
class NeuralNetwork:
    """
    Feedforward Neural Network for binary classification.
    Uses only NumPy. ReLU hidden activations, Sigmoid output.
    """

    def __init__(self, layer_dims):
        """
        Args:
            layer_dims: list of ints, e.g. [2, 64, 32, 1]
                        first element = input features
                        last element = output (1 for binary classification)
        """
        ### YOUR CODE HERE
        self.parameters = {}
        self.layer_dims = layer_dims

        numberOfLayers = len(layer_dims)
        
        rng = np.random.default_rng()

        for i in range(numberOfLayers-1):
            size = (layer_dims[i], layer_dims[i+1])

            heFactor = np.sqrt(6 / layer_dims[i])
            initialWeights = rng.standard_normal(size=size) * heFactor
            weightKey = "W" + str(i+1)
            self.parameters[weightKey] = initialWeights

            initialBias = np.zeros(shape=(1, layer_dims[i+1]))
            biasKey = "b" + str(i+1)
            self.parameters[biasKey] = initialBias

        return

    def _relu(self, Z):
        """ReLU activation."""
        ### YOUR CODE HERE
        return np.maximum(Z, 0)

    def _relu_derivative(self, Z):
        """Derivative of ReLU."""
        ### YOUR CODE HERE
        return (Z > 0).astype(float)

    def _sigmoid(self, Z):
        """Sigmoid activation."""
        ### YOUR CODE HERE
        return (1 / (1 + (np.e) ** (-Z)))

    def forward(self, X):
        """
        Forward pass through the network.

        Args:
            X: numpy array of shape (n_samples, n_features)

        Returns:
            output: numpy array of shape (n_samples, 1) — predicted probabilities

        Must store intermediate values in self._cache for use in backward().
        Cache should contain:
            'A0': input X
            'Z1', 'A1': pre-activation and activation for layer 1
            'Z2', 'A2': pre-activation and activation for layer 2
            ... etc.
        """
        ### YOUR CODE HERE
        self._cache = {}
        self._cache['A0'] = X

        noOfLayers = len(self.layer_dims)
        
        #till the last hidden layer
        for i in range(noOfLayers-2):
            Wi = self.parameters['W' + str(i+1)]
            bi = self.parameters['b' + str(i+1)]

            Aprev = self._cache['A' + str(i)]

            Znext = Aprev @ Wi + bi
            Anext = self._relu(Znext)

            self._cache['Z' + str(i+1)] = Znext
            self._cache['A' + str(i+1)] = Anext

        #final output layer
        Wi = self.parameters['W' + str(noOfLayers-1)]
        bi = self.parameters['b' + str(noOfLayers-1)]

        Aprev = self._cache['A' + str(noOfLayers-2)]

        Zoutput = Aprev @ Wi + bi
        Aoutput = self._sigmoid(Zoutput)

        self._cache['Z' + str(noOfLayers-1)] = Zoutput
        self._cache['A' + str(noOfLayers-1)] = Aoutput

        return Aoutput

    def compute_loss(self, y_pred, y_true):
        """
        Binary cross-entropy loss.

        Args:
            y_pred: (n_samples, 1)
            y_true: (n_samples, 1)

        Returns:
            loss: float
        """
        ### YOUR CODE HERE
        n_samples = y_true.shape[0]
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        bce = - (np.sum(y_true * np.log(y_pred) + (1-y_pred) * np.log(1-y_true)) / n_samples)

        return bce
        raise NotImplementedError()

    def backward(self, y_true, learning_rate=0.01):
        """
        Backpropagation: compute gradients and update weights.

        Args:
            y_true: numpy array of shape (n_samples, 1)
            learning_rate: float

        Returns:
            gradients: dict with keys 'dW1', 'db1', 'dW2', 'db2', etc.
                       for inspection/testing purposes
        """
        ### YOUR CODE HERE
        n_samples = y_true.shape[0]
        noOfLayers = len(self.layer_dims)
        gradients = {}
        
        Aoutput = self._cache['A' + str(noOfLayers-1)]
        dZ = Aoutput - y_true
        
        Aprev = self._cache['A' + str(noOfLayers - 2)]
        dW = Aprev.T @ dZ / n_samples
        db = np.sum(dZ, axis = 0, keepdims=True) / n_samples

        gradients['dW' + str(noOfLayers-1)] = dW
        gradients['db' + str(noOfLayers-1)] = db
        gradients['dZ' + str(noOfLayers-1)] = dZ

        for i in range(noOfLayers-2, 0, -1):
            Wnext = self.parameters['W' + str(i+1)]
            dZnext = gradients['dZ' + str(i+1)]
            Zi = self._cache['Z' + str(i)] #current preactivation
            Aprev = self._cache['A' + str(i-1)] #prev. activation

            dA = dZnext @ Wnext.T
            dZi = dA * self._relu_derivative(Zi)

            dWi = Aprev.T @ dZi / n_samples
            dbi = np.sum(dZi, axis = 0, keepdims=True) / n_samples

            gradients['dW' + str(i)] = dWi
            gradients['db' + str(i)] = dbi
            gradients['dZ' + str(i)] = dZi

        for i in range(1, noOfLayers):
            self.parameters['W' + str(i)] = self.parameters['W' + str(i)] - learning_rate * gradients['dW' + str(i)]
            self.parameters['b' + str(i)] = self.parameters['b' + str(i)] - learning_rate * gradients['db' + str(i)]

        return gradients

    def train(self, X, y, epochs=1000, learning_rate=0.01):
        """
        Train the network.

        Args:
            X: (n_samples, n_features)
            y: (n_samples,) or (n_samples, 1)
            epochs: int
            learning_rate: float

        Returns:
            dict with keys:
                'losses': list of float loss values per epoch
                'final_loss': float
        """
        ### YOUR CODE HERE
        if (y.ndim == 1):
            y = y.reshape(-1, 1)

        losses = []
        for i in range(epochs):
            yPred = self.forward(X)
            loss = self.compute_loss(yPred, y)
            losses.append(loss)
            self.backward(y, learning_rate)

        result = {'losses': losses, 'final_loss': losses[-1]}
        return result

    def predict(self, X):
        """
        Predict class labels.

        Args:
            X: (n_samples, n_features)

        Returns:
            predictions: (n_samples,) array of 0s and 1s
        """
        ### YOUR CODE HERE
        predictions = self.forward(X)
        return (predictions > 0.5).astype(int)

In [ ]:
# DO NOT MODIFY — Autograder test for Q3: Weight initialization

In [ ]:
# DO NOT MODIFY — Autograder test for Q3: Forward pass

In [ ]:
# DO NOT MODIFY — Autograder test for Q3: Backpropagation (gradient check)

In [ ]:
# DO NOT MODIFY — Autograder test for Q3: Training loop

In [ ]:
# DO NOT MODIFY — This cell trains your NN and computes accuracy for bracket grading
np.random.seed(42)
_X_moon, _y_moon = make_moons(n_samples=800, noise=0.3, random_state=42)
_scaler = StandardScaler()
_X_moon = _scaler.fit_transform(_X_moon)
_X_m_train, _X_m_test, _y_m_train, _y_m_test = train_test_split(_X_moon, _y_moon, test_size=0.2, random_state=42)

_nn5 = NeuralNetwork([2, 64, 32, 1])
_nn5.train(_X_m_train, _y_m_train, epochs=1000, learning_rate=0.1)
_y_m_pred = _nn5.predict(_X_m_test)
_nn_acc = accuracy_score(_y_m_test, _y_m_pred)
print(f"[Q3e] Neural Network accuracy on make_moons: {_nn_acc:.4f}")

In [ ]:
# DO NOT MODIFY — Q3 Test

In [ ]:
# DO NOT MODIFY — Q3 Test

In [ ]:
# DO NOT MODIFY — Q3 Test

# Q4: Support Vector Machine with Hinge Loss (4 marks)

Implement a linear SVM using subgradient descent on the hinge loss. Then implement an RBF kernel function.

**sklearn is NOT allowed for the SVM implementation.** You may use sklearn for data generation and evaluation metrics only.

**Mathematical Reference:**

Hinge Loss (primal SVM objective):
$\mathcal{L}(w, b) = \frac{1}{2}||w||^2 + C \sum_{i=1}^{m} \max(0, 1 - y_i(w^T x_i + b))$

Subgradient for $w$:
$\nabla_w \mathcal{L} = w - C \sum_{i: y_i(w^T x_i + b) < 1} y_i x_i$

Subgradient for $b$:
$\nabla_b \mathcal{L} = -C \sum_{i: y_i(w^T x_i + b) < 1} y_i$

RBF Kernel:
$K(x_i, x_j) = \exp\left(-\gamma ||x_i - x_j||^2\right)$

**Note:** The labels must be +1 and -1 (not 0 and 1) for the hinge loss formulation.

### Grading:
1. `hinge_loss` function (0.5 marks)
2. `compute_svm_gradients` function (0.5 marks)
3. `SVMClassifier.fit` — convergence (1 mark)
4. `rbf_kernel` function (0.5 marks)
5. `SVMClassifier.predict` + accuracy bracket (1.5 marks):

| Accuracy | Score |
|----------|-------|
| > 90%    | 1.5/1.5 (0.5 + 0.5 + 0.5) |
| 84–90%   | 1.0/1.5 (0.5 + 0.5)       |
| 78–84%   | 0.5/1.5 (0.5)             |
| < 78%    | 0/1.5                     |

In [ ]:
def hinge_loss(X, y, w, b, C=1.0):
    """
    Compute SVM hinge loss.

    Args:
        X: (n_samples, n_features)
        y: (n_samples,) with values in {-1, +1}
        w: (n_features,)
        b: float
        C: regularization parameter

    Returns:
        loss: float
    """
    ### YOUR CODE HERE
    y1 = np.maximum(0, 1 - y * (X @ w + b))
    return np.sum(y1) * C + (0.5 * np.sum(w ** 2))


def compute_svm_gradients(X, y, w, b, C=1.0):
    """
    Compute subgradients of hinge loss w.r.t. w and b.

    Args:
        X: (n_samples, n_features)
        y: (n_samples,) with values in {-1, +1}
        w: (n_features,)
        b: float
        C: regularization parameter

    Returns:
        dw: (n_features,) gradient w.r.t. w
        db: float, gradient w.r.t. b
    """
    ### YOUR CODE HERE
    margins = y * (X @ w + b)
    X_violated = X[ margins < 1]
    y_violated = y[ margins < 1]
    dw = w - C * (X_violated.T @ y_violated)
    db = -C * np.sum(y_violated)
    return dw, db


def rbf_kernel(X1, X2, gamma=1.0):
    """
    Compute RBF (Gaussian) kernel matrix.

    Args:
        X1: (n1, n_features)
        X2: (n2, n_features)
        gamma: float, kernel bandwidth parameter

    Returns:
        K: (n1, n2) kernel matrix
    """
    ### YOUR CODE HERE
    X1sq = np.sum(X1 ** 2, axis=1, keepdims=True)
    X2sq = np.sum(X2 ** 2, axis=1, keepdims=True)
    return np.exp(-gamma * (X1sq + X2sq.T - 2 * (X1 @ X2.T)))


class SVMClassifier:
    """Linear SVM using subgradient descent on hinge loss."""

    def __init__(self, C=1.0, learning_rate=0.01, epochs=1000):
        ### YOUR CODE HERE
        self.C = C
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.w = None
        self.b = None
        return

    def fit(self, X, y):
        """
        Train SVM using subgradient descent.
        Labels y must be in {-1, +1}.

        Args:
            X: (n_samples, n_features)
            y: (n_samples,) values in {-1, +1}

        Returns:
            dict with key 'losses': list of loss values per epoch
        """
        ### YOUR CODE HERE
        y = np.where(y <= 0, -1, 1)
        n_samples, n_features = X.shape
        self.w = np.random.uniform(size=n_features, low=0.0, high=1.0)
        self.b = 0.0

        losses = []
        for i in range(self.epochs):
            dw, db = compute_svm_gradients(X, y, self.w, self.b, self.C)
            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

            current_loss = hinge_loss(X, y, self.w, self.b, self.C)
            losses.append(current_loss)

        result = {'losses':losses}
        return result


    def predict(self, X):
        """
        Predict class labels {-1, +1}.

        Args:
            X: (n_samples, n_features)

        Returns:
            predictions: (n_samples,) array of -1 and +1
        """
        ### YOUR CODE HERE
        predictions = X @ self.w + self.b
        predictions = np.where(predictions>0, 1, -1)
        return predictions

In [ ]:
# DO NOT MODIFY — Autograder test for Q4: Hinge loss

In [ ]:
# DO NOT MODIFY — Autograder test for Q4: Gradients (numerical check)

In [ ]:
# DO NOT MODIFY — Autograder test for Q4: Fit + convergence

In [ ]:
# DO NOT MODIFY — Autograder test for Q4: RBF kernel

In [ ]:
# DO NOT MODIFY — This cell trains your SVM and computes accuracy for bracket grading
np.random.seed(42)
_X_svm_acc, _y_svm_acc = make_moons(n_samples=600, noise=0.25, random_state=7)
_y_svm_acc = 2 * _y_svm_acc - 1  # Convert 0/1 to -1/+1
_scaler_svm = StandardScaler()
_X_svm_acc = _scaler_svm.fit_transform(_X_svm_acc)
_X_s_train, _X_s_test, _y_s_train, _y_s_test = train_test_split(_X_svm_acc, _y_svm_acc, test_size=0.2, random_state=42)

_svm_final = SVMClassifier(C=0.5, learning_rate=0.0005, epochs=2000)
_svm_final.fit(_X_s_train, _y_s_train)
_y_s_pred = _svm_final.predict(_X_s_test)
_svm_acc = accuracy_score(_y_s_test, _y_s_pred)
print(f"[Q4e] SVM accuracy: {_svm_acc:.4f}")

In [ ]:
# DO NOT MODIFY — Q4 Test

In [ ]:
# DO NOT MODIFY — Q4 Test

In [ ]:
# DO NOT MODIFY — Q4 Test

---